# Predicting Geomagnetic Storms from Solar Events and Solar Images

**BUDA 460 – Artificial Intelligence and Machine Learning for Business · Individual project**

This notebook predicts whether a geomagnetic storm (Kp ≥ 5, NOAA level G1 or higher) will **begin within the next 48 hours**, using the previous **14 days** of:

1. **Solar events** from NASA's DONKI space weather database (flares, coronal mass ejections, particle events, shocks, high-speed streams), and
2. **Daily images of the Sun** from NASA's Helioviewer API (SDO/AIA 193 Å and SDO/HMI magnetogram).

It compares simple baselines (logistic regression, XGBoost) with deep learning models (LSTM, 1D-CNN, an image CNN, and a CNN + LSTM fusion model), and tests four ways of handling the class imbalance (storm days are rare).

**How to run:** run all cells top to bottom. The first run downloads the data (DONKI takes a few minutes; the images can take 1–3 hours). Everything is cached in `data/`, so later runs are fast. All figures and result tables are saved in `outputs/`.

## 1. Setup and configuration

In [ ]:
import os, re, json, time, math, random, warnings
from pathlib import Path
from datetime import date, datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score,
                             f1_score, fbeta_score, confusion_matrix, precision_recall_curve)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

try:
    from tqdm.auto import tqdm
except ImportError:          # tqdm is optional
    def tqdm(x, **k): return x

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# ----------------------------- configuration -----------------------------
START_DATE   = "2010-06-01"                 # SDO began science operations in May 2010
END_DATE     = (date.today() - timedelta(days=3)).isoformat()
WINDOW_DAYS  = 14                           # days of history the model sees
HORIZON_DAYS = 2                            # predict a storm onset in the next 48 hours
KP_THRESHOLD = 5                            # Kp >= 5 = NOAA G1 (minor storm) or stronger
TRAIN_END    = "2020-12-31"                 # train: 2010-2020
VAL_END      = "2022-12-31"                 # validation: 2021-2022, test: 2023 onwards

NASA_API_KEY = os.environ.get("NASA_API_KEY", "DEMO_KEY")   # only needed for the api.nasa.gov fallback
USE_IMAGES   = True                         # set False to run the event-only models
IMAGE_SIZE   = 256                          # pixels (downloaded); resized to 224 for the CNN
IMAGE_SCALE  = 9.0                          # arcseconds per pixel: 256 px x 9 = whole solar disk
IMAGE_WORKERS = 6                           # parallel image downloads
CHANNELS = {                                # Helioviewer layer strings
    "aia193":  "[SDO,AIA,AIA,193,1,100]",            # extreme ultraviolet: coronal holes
    "hmi_mag": "[SDO,HMI,HMI,magnetogram,1,100]",    # magnetic field of sunspot regions
}
PCA_COMPONENTS = 16                         # image summary numbers per channel for the fusion model
N_SEEDS      = 3                            # deep models are averaged over this many random seeds
COST_MISSED_STORM = 10                      # illustrative business costs (relative units)
COST_FALSE_ALARM  = 1

DATA_DIR = Path("data"); RAW_DIR = DATA_DIR / "raw"; IMG_DIR = DATA_DIR / "images"
OUT_DIR = Path("outputs"); FIG_DIR = OUT_DIR / "figures"
for d in [RAW_DIR, IMG_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(42)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")             # Apple Silicon (M1/M2/M3)
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE, "| Data range:", START_DATE, "to", END_DATE)

## 2. Download solar events from NASA DONKI

DONKI's web service moved to `ccmc.gsfc.nasa.gov` in September 2026; the old `kauai` address and `api.nasa.gov` are kept as fallbacks. Data is requested one year at a time and cached as JSON in `data/raw/`.

In [ ]:
DONKI_BASES = ["https://ccmc.gsfc.nasa.gov/DONKI/WS/get",
               "https://kauai.ccmc.gsfc.nasa.gov/DONKI/WS/get",
               "https://api.nasa.gov/DONKI"]
EVENT_TYPES = ["FLR", "CME", "SEP", "IPS", "HSS", "GST"]

def fetch_donki(event_type, start, end):
    """Return the list of DONKI records of one type between two dates (inclusive)."""
    errors = []
    for base in DONKI_BASES:
        params = {"startDate": start, "endDate": end}
        if "api.nasa.gov" in base:
            params["api_key"] = NASA_API_KEY
        for attempt in range(3):
            try:
                r = requests.get(f"{base}/{event_type}", params=params, timeout=90)
                if r.status_code == 200:
                    text = r.text.strip()
                    return json.loads(text) if text and text != "null" else []
                errors.append(f"{base}: HTTP {r.status_code}")
                if r.status_code in (429, 500, 502, 503, 504):
                    time.sleep(5 * (attempt + 1)); continue
                break
            except (requests.RequestException, ValueError) as e:
                errors.append(f"{base}: {type(e).__name__}")
                time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"Could not download {event_type} {start}..{end}: {errors[-3:]}")

def load_donki(event_type):
    start, end = pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)
    records = []
    for year in range(start.year, end.year + 1):
        path = RAW_DIR / f"{event_type}_{year}.json"
        y_start = max(start, pd.Timestamp(f"{year}-01-01")).strftime("%Y-%m-%d")
        y_end = min(end, pd.Timestamp(f"{year}-12-31")).strftime("%Y-%m-%d")
        is_current_year = year == end.year
        fresh = path.exists() and (not is_current_year or time.time() - path.stat().st_mtime < 86400)
        if fresh:      # past years never change; the current year is refreshed once a day
            data = json.loads(path.read_text())
        else:
            data = fetch_donki(event_type, y_start, y_end)
            path.write_text(json.dumps(data))
            time.sleep(0.5)
        records.extend(data or [])
    return records

raw = {}
for et in EVENT_TYPES:
    raw[et] = load_donki(et)
    print(f"{et}: {len(raw[et]):,} records")

## 3. Download daily solar images from NASA Helioviewer

One full-disk image per channel per day at 12:00 UTC, 256 × 256 pixels, saved as PNG in `data/images/<channel>/`. Downloads are parallel and resumable: if the run stops, re-running this cell continues where it left off.

In [ ]:
HV_URL = "https://api.helioviewer.org/v2/takeScreenshot/"
ALL_DAYS = pd.date_range(START_DATE, END_DATE, freq="D")

def image_path(channel, day):
    return IMG_DIR / channel / f"{day:%Y-%m-%d}.png"

def fetch_image(channel, layer, day):
    path = image_path(channel, day)
    if path.exists() and path.stat().st_size > 1000:
        return True
    params = {"date": f"{day:%Y-%m-%d}T12:00:00Z", "imageScale": IMAGE_SCALE, "layers": layer,
              "x0": 0, "y0": 0, "width": IMAGE_SIZE, "height": IMAGE_SIZE,
              "display": "true", "watermark": "false", "events": ""}
    for attempt in range(3):
        try:
            r = requests.get(HV_URL, params=params, timeout=120)
            if r.status_code == 200 and r.headers.get("content-type", "").startswith("image"):
                path.write_bytes(r.content)
                return True
        except requests.RequestException:
            pass
        time.sleep(2 * (attempt + 1))
    return False

if USE_IMAGES:
    for channel, layer in CHANNELS.items():
        (IMG_DIR / channel).mkdir(parents=True, exist_ok=True)
        todo = [d for d in ALL_DAYS if not image_path(channel, d).exists()]
        print(f"{channel}: {len(ALL_DAYS) - len(todo):,} cached, {len(todo):,} to download")
        failed = []
        with ThreadPoolExecutor(max_workers=IMAGE_WORKERS) as pool:
            futures = {pool.submit(fetch_image, channel, layer, d): d for d in todo}
            for f in tqdm(as_completed(futures), total=len(futures), desc=channel):
                if not f.result():
                    failed.append(futures[f])
        print(f"{channel}: {len(failed)} days could not be downloaded (handled as missing later)")

## 4. Parse the event records into tables

In [ ]:
FLARE_SCALE = {"A": 1e-8, "B": 1e-7, "C": 1e-6, "M": 1e-5, "X": 1e-4}

def flare_flux(cls):
    """'M2.5' -> 2.5e-5 W/m^2 (peak X-ray flux)."""
    if not isinstance(cls, str) or not cls:
        return np.nan
    m = re.match(r"([ABCMX])\s*([\d.]+)?", cls.strip().upper())
    if not m:
        return np.nan
    return FLARE_SCALE[m.group(1)] * float(m.group(2) or 1.0)

def location_lon(loc):
    """'N12W25' -> +25 (west), 'S05E40' -> -40 (east). Longitude 0 faces Earth."""
    if not isinstance(loc, str):
        return np.nan
    m = re.search(r"([NS])(\d+)([EW])(\d+)", loc.upper())
    if not m:
        return np.nan
    return float(m.group(4)) * (1 if m.group(3) == "W" else -1)

def to_time(s):
    return pd.to_datetime(s, utc=True, errors="coerce").tz_convert(None) if s else pd.NaT

flr = pd.DataFrame([{
    "time": to_time(r.get("peakTime") or r.get("beginTime")),
    "cls": (r.get("classType") or "?")[:1].upper(),
    "flux": flare_flux(r.get("classType")),
    "lon": location_lon(r.get("sourceLocation")),
} for r in raw["FLR"]])

def best_analysis(rec):
    an = rec.get("cmeAnalyses") or []
    best = [a for a in an if a.get("isMostAccurate")] or an
    return best[0] if best else {}

cme = pd.DataFrame([{
    "time": to_time(r.get("startTime")),
    "speed": best_analysis(r).get("speed"),
    "half_angle": best_analysis(r).get("halfAngle"),
    "lat": best_analysis(r).get("latitude"),
    "lon": best_analysis(r).get("longitude"),
    "note": r.get("note") or "",
} for r in raw["CME"]])
for c in ["speed", "half_angle", "lat", "lon"]:
    cme[c] = pd.to_numeric(cme[c], errors="coerce")
cme["earth_directed"] = (cme["lon"].abs() <= 30) & (cme["lat"].abs() <= 30)

sep = pd.DataFrame([{"time": to_time(r.get("eventTime"))} for r in raw["SEP"]])
ips = pd.DataFrame([{"time": to_time(r.get("eventTime")),
                     "at_earth": str(r.get("location", "")).strip().lower() == "earth"} for r in raw["IPS"]])
hss = pd.DataFrame([{"time": to_time(r.get("eventTime"))} for r in raw["HSS"]])

kp_rows, storm_rows = [], []
for r in raw["GST"]:
    kps = [(to_time(k.get("observedTime")), float(k.get("kpIndex") or 0)) for k in (r.get("allKpIndex") or [])]
    kp_rows += [{"time": t, "kp": kp} for t, kp in kps if pd.notna(t)]
    strong = sorted(t for t, kp in kps if kp >= KP_THRESHOLD and pd.notna(t))
    if strong:   # onset = first 3-hour period with Kp >= threshold
        storm_rows.append({"onset": strong[0], "max_kp": max(kp for _, kp in kps)})
kp = pd.DataFrame(kp_rows, columns=["time", "kp"])
storms = pd.DataFrame(storm_rows, columns=["onset", "max_kp"]).sort_values("onset").reset_index(drop=True)

for name, df in [("flares", flr), ("CMEs", cme), ("SEPs", sep), ("shocks", ips), ("HSS", hss), ("storms (Kp>=5)", storms)]:
    print(f"{name:15s} {len(df):6,d}")

## 5. Exploratory analysis

In [ ]:
def per_year(df, col="time"):
    return df[col].dt.year.value_counts().sort_index() if len(df) else pd.Series(dtype=int)

eda = pd.DataFrame({"flares": per_year(flr), "CMEs": per_year(cme), "SEPs": per_year(sep),
                    "shocks": per_year(ips), "storms": per_year(storms, "onset")}).fillna(0).astype(int)
eda.index.name = "year"
display(eda)

fig, ax = plt.subplots(figsize=(10, 4))
ax.axvspan(2020.5, 2022.5, color="grey", alpha=0.15, label="validation years", zorder=0)
ax.axvspan(2022.5, eda.index.max() + 0.5, color="orange", alpha=0.15, label="test years", zorder=0)
ax.bar(eda.index, eda["storms"], color="#2f6db5", zorder=2)
ax.set_xticks(eda.index); ax.set_xticklabels(eda.index, rotation=45)
ax.set_title("Geomagnetic storms (Kp ≥ 5) per year in DONKI")
ax.set_xlabel("Year"); ax.set_ylabel("Storm onsets")
ax.legend(loc="upper left", frameon=False); plt.tight_layout(); plt.savefig(FIG_DIR / "storms_per_year.png", dpi=150); plt.show()

## 6. Daily features

Each day gets one row of numbers summarising that day's solar activity. Counts and speeds are log-transformed, then standardised with a scaler fitted on the **training years only** (no information from later years leaks in).

In [ ]:
days = pd.DataFrame(index=ALL_DAYS)

def daily(df, func, col="time"):
    if len(df) == 0:
        return pd.Series(0.0, index=ALL_DAYS)
    g = df.dropna(subset=[col]).groupby(df[col].dt.floor("D"))
    return func(g).reindex(ALL_DAYS).fillna(0.0)

for c in "CMX":
    days[f"flares_{c}"] = daily(flr[flr["cls"] == c], lambda g: g.size())
days["flare_max_log_flux"] = daily(flr, lambda g: np.log10(g["flux"].max())).replace(0, np.nan).fillna(-8)
days["flares_earth_facing_MX"] = daily(flr[flr["cls"].isin(["M", "X"]) & (flr["lon"].abs() <= 30)], lambda g: g.size())
days["cmes"] = daily(cme, lambda g: g.size())
days["cme_max_speed"] = daily(cme, lambda g: g["speed"].max())
days["cme_max_half_angle"] = daily(cme, lambda g: g["half_angle"].max())
days["cmes_earth_directed"] = daily(cme[cme["earth_directed"]], lambda g: g.size())
days["cme_earth_max_speed"] = daily(cme[cme["earth_directed"]], lambda g: g["speed"].max())
days["cmes_missing_analysis"] = daily(cme[cme["speed"].isna()], lambda g: g.size())
days["seps"] = daily(sep, lambda g: g.size())
days["shocks_at_earth"] = daily(ips[ips["at_earth"]], lambda g: g.size())
days["hss"] = daily(hss, lambda g: g.size())
days["kp_max_today"] = daily(kp, lambda g: g["kp"].max())
days["storm_today"] = (days["kp_max_today"] >= KP_THRESHOLD).astype(float)

x_day = pd.Series(days.index, index=days.index).where(days["flares_X"] > 0).ffill()
days["days_since_X_flare"] = ((days.index - x_day).dt.days).fillna(365).clip(upper=365)

EVENT_FEATURES = list(days.columns)
log_cols = [c for c in EVENT_FEATURES if c not in ("flare_max_log_flux", "storm_today", "kp_max_today")]
feat = days.copy()
feat[log_cols] = np.log1p(feat[log_cols].clip(lower=0))

train_days = feat.index <= TRAIN_END
scaler = StandardScaler().fit(feat.loc[train_days, EVENT_FEATURES])
feat_scaled = pd.DataFrame(scaler.transform(feat[EVENT_FEATURES]), index=feat.index, columns=EVENT_FEATURES)
print(f"{len(EVENT_FEATURES)} event features per day:", ", ".join(EVENT_FEATURES))
display(days.describe().T[["mean", "std", "max"]].round(2))

## 7. Labels: does a storm begin in the next 48 hours?

A sample uses the 14 days ending on day *d*. Its label is **1** if a storm (Kp ≥ 5) begins on day *d*+1 or *d*+2, otherwise **0**.

In [ ]:
onset_days = set(storms["onset"].dt.floor("D"))
onset_flag = pd.Series([d in onset_days for d in ALL_DAYS], index=ALL_DAYS).astype(int)
label = sum(onset_flag.shift(-k) for k in range(1, HORIZON_DAYS + 1)).clip(upper=1)
days["label"] = label

def split_of(d):
    if d <= pd.Timestamp(TRAIN_END) - pd.Timedelta(days=HORIZON_DAYS): return "train"
    if pd.Timestamp(TRAIN_END) < d <= pd.Timestamp(VAL_END) - pd.Timedelta(days=HORIZON_DAYS): return "val"
    if d > pd.Timestamp(VAL_END): return "test"
    return "gap"     # a 2-day gap stops labels crossing a split boundary

days["split"] = [split_of(d) for d in ALL_DAYS]
valid = np.arange(WINDOW_DAYS - 1, len(ALL_DAYS) - HORIZON_DAYS)
sample_idx = {s: valid[(days["split"].values[valid] == s)] for s in ["train", "val", "test"]}

balance = pd.DataFrame({s: {"samples": len(ix), "storm (1)": int(days["label"].values[ix].sum()),
                            "storm share": days["label"].values[ix].mean()} for s, ix in sample_idx.items()}).T
balance["storm share"] = (balance["storm share"] * 100).round(1).astype(str) + "%"
display(balance)

fig, ax = plt.subplots(figsize=(6, 3.5))
counts = days.iloc[valid]["label"].value_counts().sort_index()
ax.bar(["No storm (0)", "Storm in next 48 h (1)"], counts.values, color=["#9aa5b1", "#c0392b"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,} ({v / counts.sum():.1%})", ha="center", va="bottom")
ax.set_ylim(0, counts.max() * 1.15)
ax.set_title("Class imbalance in the labels"); ax.set_ylabel("Samples (days)")
plt.tight_layout(); plt.savefig(FIG_DIR / "class_imbalance.png", dpi=150); plt.show()

## 8. Image features with a pretrained CNN

Each day's two images go through **ResNet-18 pretrained on ImageNet** (a convolutional neural network). Its last layer gives 512 numbers per image that describe its shapes and textures (dark coronal holes, bright active regions, complex magnetic patterns). Missing days reuse the previous day's image when it is at most one day old; otherwise the numbers are zero and a "missing" flag is set.

In [ ]:
import torchvision
from torchvision import transforms

def load_cnn():
    try:
        weights = torchvision.models.ResNet18_Weights.DEFAULT
        model = torchvision.models.resnet18(weights=weights)
        pretrained = True
    except Exception as e:      # e.g. no internet for the weights download
        print("WARNING: pretrained weights unavailable, using random weights:", type(e).__name__)
        model = torchvision.models.resnet18(weights=None)
        pretrained = False
    model.fc = nn.Identity()
    return model.eval().to(DEVICE), pretrained

PREPROCESS = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

@torch.no_grad()
def embed_channel(model, channel, batch_size=64):
    cache = DATA_DIR / f"embeddings_{channel}.npz"
    if cache.exists():
        z = np.load(cache)
        if len(z["emb"]) == len(ALL_DAYS):
            return z["emb"], z["ok"]
    emb = np.zeros((len(ALL_DAYS), 512), dtype=np.float32)
    ok = np.zeros(len(ALL_DAYS), dtype=bool)
    paths = [image_path(channel, d) for d in ALL_DAYS]
    for start in tqdm(range(0, len(paths), batch_size), desc=f"embedding {channel}"):
        batch, idx = [], []
        for i in range(start, min(start + batch_size, len(paths))):
            try:
                batch.append(PREPROCESS(Image.open(paths[i]).convert("RGB"))); idx.append(i)
            except Exception:
                pass
        if batch:
            emb[idx] = model(torch.stack(batch).to(DEVICE)).cpu().numpy()
            ok[idx] = True
    np.savez(cache, emb=emb, ok=ok)
    return emb, ok

if USE_IMAGES:
    cnn, PRETRAINED_OK = load_cnn()
    img_emb, img_ok = {}, {}
    for ch in CHANNELS:
        e, ok = embed_channel(cnn, ch)
        for i in np.where(~ok)[0]:          # fill a missing day from the day before
            if i > 0 and ok[i - 1]:
                e[i] = e[i - 1]
        img_emb[ch], img_ok[ch] = e, ok
        print(f"{ch}: images available for {ok.mean():.1%} of days")
    img_missing = (~np.all(np.stack([img_ok[c] for c in CHANNELS]), axis=0)).astype(np.float32)

    # PCA per channel, fitted on training days only -> compact image summary for the fusion model
    img_pca = []
    for ch in CHANNELS:
        p = PCA(n_components=PCA_COMPONENTS, random_state=42).fit(img_emb[ch][train_days])
        comp = p.transform(img_emb[ch])
        comp = (comp - comp[train_days].mean(0)) / (comp[train_days].std(0) + 1e-6)
        img_pca.append(comp)
        print(f"{ch}: {PCA_COMPONENTS} components explain {p.explained_variance_ratio_.sum():.1%} of image variance")
    IMG_DAILY = np.hstack(img_pca + [img_missing[:, None]]).astype(np.float32)
    IMG_FULL = np.hstack([img_emb[c] for c in CHANNELS]).astype(np.float32)
    IMG_FULL = (IMG_FULL - IMG_FULL[train_days].mean(0)) / (IMG_FULL[train_days].std(0) + 1e-6)
else:
    PRETRAINED_OK = None

**Example images.** One storm-free day and the day before a storm, in both channels.

In [ ]:
if USE_IMAGES:
    storm_eve = days.index[(days["label"] == 1) & (days["split"] == "test")]
    quiet = days.index[(days["label"] == 0) & (days["split"] == "test")]
    pick = [d for d in [quiet[len(quiet) // 2] if len(quiet) else None,
                        storm_eve[len(storm_eve) // 2] if len(storm_eve) else None] if d is not None]
    fig, axes = plt.subplots(len(CHANNELS), len(pick), figsize=(3.2 * len(pick), 3.2 * len(CHANNELS)), squeeze=False)
    for r, ch in enumerate(CHANNELS):
        for c, d in enumerate(pick):
            ax = axes[r][c]; ax.axis("off")
            try:
                ax.imshow(Image.open(image_path(ch, d)))
            except Exception:
                ax.text(0.5, 0.5, "missing", ha="center")
            tag = "storm within 48 h" if days.loc[d, "label"] == 1 else "no storm"
            ax.set_title(f"{ch} · {d:%Y-%m-%d}\n{tag}", fontsize=9)
    plt.tight_layout(); plt.savefig(FIG_DIR / "example_images.png", dpi=150); plt.show()

## 9. Build 14-day windows and split by time

Train 2010–2020, validate 2021–2022, test 2023 onwards. The split is by date, never shuffled, so the model is always tested on years it has never seen.

In [ ]:
EV = feat_scaled.values.astype(np.float32)
y_all = days["label"].values.astype(np.float32)

def windows(daily_matrix, idx):
    return np.stack([daily_matrix[i - WINDOW_DAYS + 1:i + 1] for i in idx])

data = {}
for s, idx in sample_idx.items():
    d = {"idx": idx, "y": y_all[idx], "date": ALL_DAYS[idx],
         "seq_ev": windows(EV, idx),
         "persistence": days["storm_today"].values[idx]}
    d["flat_ev"] = d["seq_ev"].reshape(len(idx), -1)
    if USE_IMAGES:
        d["seq_fusion"] = windows(np.hstack([EV, IMG_DAILY]), idx)
        d["img_today"] = IMG_FULL[idx]
    data[s] = d
    print(f"{s:5s}: {len(idx):5,d} samples | sequence shape {d['seq_ev'].shape}"
          + (f" | fusion {d['seq_fusion'].shape}" if USE_IMAGES else ""))

## 10. Imbalance tools and evaluation metrics

* **Resampling** (training only): storm windows are drawn more often, with small random noise added so the model does not memorise exact copies.
* **Loss**: standard, class-weighted (a missed storm costs `neg/pos` times more), or **focal loss** (focuses training on hard examples).
* **Threshold**: chosen on the validation years to maximise **F2**, which counts recall twice as much as precision, because missing a storm is costlier than a false alarm.
* **Metrics**: PR-AUC (primary), storm recall, precision, F1, F2, macro F1, ROC-AUC. Accuracy and weighted F1 are not used: they hide failure on the rare class.

In [ ]:
def tune_threshold(y, p):
    """Threshold on validation probabilities that maximises F2."""
    best_t, best_f2 = 0.5, -1
    for t in np.unique(np.quantile(p, np.linspace(0.5, 0.999, 200))):
        f2 = fbeta_score(y, p >= t, beta=2, zero_division=0)
        if f2 > best_f2:
            best_t, best_f2 = float(t), f2
    return best_t

def evaluate(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "PR-AUC": average_precision_score(y, p) if y.sum() else np.nan,
        "ROC-AUC": roc_auc_score(y, p) if 0 < y.sum() < len(y) and len(np.unique(p)) > 1 else 0.5,
        "Recall": recall_score(y, pred, zero_division=0),
        "Precision": precision_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "F2": fbeta_score(y, pred, beta=2, zero_division=0),
        "Macro F1": f1_score(y, pred, average="macro", zero_division=0),
        "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn), "Threshold": thr,
    }

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__(); self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, y):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, y, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * y + (1 - p) * (1 - y)
        a_t = self.alpha * y + (1 - self.alpha) * (1 - y)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

def make_loss(kind, y_train):
    if kind == "weighted":
        pw = torch.tensor([(1 - y_train).sum() / max(y_train.sum(), 1)], device=DEVICE)
        return nn.BCEWithLogitsLoss(pos_weight=pw)
    if kind == "focal":
        return FocalLoss()
    return nn.BCEWithLogitsLoss()

## 11. Deep learning models

In [ ]:
class LSTMNet(nn.Module):
    """Reads the 14-day sequence in order; used for events-only and for the CNN + LSTM fusion."""
    def __init__(self, n_features, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden, 1))
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.head(h[-1]).squeeze(-1)

class CNN1DNet(nn.Module):
    """1D convolutions slide across the 14 days to detect short bursts of activity."""
    def __init__(self, n_features, channels=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_features, channels, 3, padding=1), nn.ReLU(),
            nn.Conv1d(channels, channels, 3, padding=1), nn.ReLU(),
            nn.AdaptiveMaxPool1d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(channels, 1))
    def forward(self, x):
        return self.net(x.transpose(1, 2)).squeeze(-1)

class ImageHead(nn.Module):
    """Classifier on top of the pretrained ResNet-18 features of the latest day's two images."""
    def __init__(self, n_in, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(0.2), nn.Linear(n_in, hidden), nn.ReLU(),
                                 nn.Dropout(0.3), nn.Linear(hidden, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

@torch.no_grad()
def predict(model, X, batch=1024):
    model.eval()
    out = [torch.sigmoid(model(torch.tensor(X[i:i + batch], device=DEVICE))).cpu().numpy()
           for i in range(0, len(X), batch)]
    return np.concatenate(out)

def train_one(make_model, Xtr, ytr, Xva, yva, loss="weighted", oversample=False, seed=0,
              epochs=60, lr=1e-3, batch=128, patience=10, noise=0.05):
    set_seed(seed)
    model = make_model().to(DEVICE)
    crit = make_loss(loss, ytr)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    ds = TensorDataset(torch.tensor(Xtr), torch.tensor(ytr))
    if oversample:
        w = np.where(ytr == 1, (1 - ytr).sum() / max(ytr.sum(), 1), 1.0)
        sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), len(ytr), replacement=True,
                                        generator=torch.Generator().manual_seed(seed))
        loader = DataLoader(ds, batch_size=batch, sampler=sampler)
    else:
        loader = DataLoader(ds, batch_size=batch, shuffle=True, generator=torch.Generator().manual_seed(seed))
    best, best_state, wait = -1, None, 0
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if oversample and noise:   # jitter so repeated storm windows are not exact copies
                xb = xb + noise * torch.randn_like(xb) * yb.view(-1, *[1] * (xb.dim() - 1))
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        score = average_precision_score(yva, predict(model, Xva))
        if score > best:
            best, wait = score, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model

def train_ensemble(make_model, key, loss="weighted", oversample=False):
    """Train N_SEEDS models and average their probabilities (more stable on small data)."""
    tr, va, te = data["train"], data["val"], data["test"]
    p_val, p_test = [], []
    for seed in range(N_SEEDS):
        m = train_one(make_model, tr[key], tr["y"], va[key], va["y"], loss=loss, oversample=oversample, seed=seed)
        p_val.append(predict(m, va[key])); p_test.append(predict(m, te[key]))
    return np.mean(p_val, 0), np.mean(p_test, 0)

## 12. Train all models

Every model uses its standard imbalance fix (class weights) and a threshold tuned on the validation years. Scores are on the **test years only**.

In [ ]:
y_val, y_test = data["val"]["y"], data["test"]["y"]
preds = {}   # model name -> (validation probabilities, test probabilities)

# Simple baselines
preds["Always 'no storm'"] = (np.zeros_like(y_val), np.zeros_like(y_test))
preds["Persistence (storm today)"] = (data["val"]["persistence"], data["test"]["persistence"])

# Logistic regression
lr_model = LogisticRegression(class_weight="balanced", max_iter=3000, C=0.1)
lr_model.fit(data["train"]["flat_ev"], data["train"]["y"])
preds["Logistic regression"] = (lr_model.predict_proba(data["val"]["flat_ev"])[:, 1],
                                lr_model.predict_proba(data["test"]["flat_ev"])[:, 1])

# XGBoost (falls back to scikit-learn gradient boosting if xgboost is not installed)
ytr = data["train"]["y"]
spw = float((1 - ytr).sum() / max(ytr.sum(), 1))
try:
    from xgboost import XGBClassifier
    gb = XGBClassifier(n_estimators=600, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.6,
                       min_child_weight=3, scale_pos_weight=spw, eval_metric="aucpr",
                       early_stopping_rounds=50, random_state=42, n_jobs=-1)
    gb.fit(data["train"]["flat_ev"], ytr, eval_set=[(data["val"]["flat_ev"], y_val)], verbose=False)
    gb_name = "XGBoost"
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    gb = HistGradientBoostingClassifier(class_weight="balanced", max_depth=4, random_state=42)
    gb.fit(data["train"]["flat_ev"], ytr)
    gb_name = "Gradient boosting"
preds[gb_name] = (gb.predict_proba(data["val"]["flat_ev"])[:, 1], gb.predict_proba(data["test"]["flat_ev"])[:, 1])

# Deep models on event sequences
n_ev = data["train"]["seq_ev"].shape[-1]
preds["LSTM (events)"] = train_ensemble(lambda: LSTMNet(n_ev), "seq_ev")
preds["1D-CNN (events)"] = train_ensemble(lambda: CNN1DNet(n_ev), "seq_ev")

# Deep models with images
if USE_IMAGES:
    n_img = data["train"]["img_today"].shape[-1]
    preds["Image CNN (ResNet-18 features)"] = train_ensemble(lambda: ImageHead(n_img), "img_today")
    n_fu = data["train"]["seq_fusion"].shape[-1]
    preds["CNN + LSTM fusion"] = train_ensemble(lambda: LSTMNet(n_fu), "seq_fusion")

rows = {}
for name, (pv, pt) in preds.items():
    thr = 0.5 if name.startswith(("Always", "Persistence")) else tune_threshold(y_val, pv)
    rows[name] = evaluate(y_test, pt, thr)
results = pd.DataFrame(rows).T
results_view = results[["PR-AUC", "Recall", "Precision", "F1", "F2", "Macro F1", "ROC-AUC", "TP", "FN", "FP"]]
display(results_view.style.format({c: "{:.3f}" for c in ["PR-AUC", "Recall", "Precision", "F1", "F2", "Macro F1", "ROC-AUC"]})
        .highlight_max(subset=["PR-AUC", "Recall", "F1", "F2"], color="#d6eaf8"))
print(f"Test-set storm share (PR-AUC of a random guess): {y_test.mean():.3f}")

## 13. Imbalance ablation on the best deep model

The same model is trained five ways to show what each imbalance technique adds. Variants A–D use the default 0.5 threshold; E combines the best of B–D with the tuned threshold.

In [ ]:
deep_names = [n for n in preds if n in ("LSTM (events)", "1D-CNN (events)", "Image CNN (ResNet-18 features)", "CNN + LSTM fusion")]
best_deep = max(deep_names, key=lambda n: average_precision_score(y_val, preds[n][0]))
spec = {"LSTM (events)": (lambda: LSTMNet(n_ev), "seq_ev"),
        "1D-CNN (events)": (lambda: CNN1DNet(n_ev), "seq_ev")}
if USE_IMAGES:
    spec["Image CNN (ResNet-18 features)"] = (lambda: ImageHead(n_img), "img_today")
    spec["CNN + LSTM fusion"] = (lambda: LSTMNet(n_fu), "seq_fusion")
make_model, key = spec[best_deep]
print("Ablation model (best validation PR-AUC):", best_deep)

variants = {"A: no fix": dict(loss="standard", oversample=False),
            "B: oversampling": dict(loss="standard", oversample=True),
            "C: class weights": dict(loss="weighted", oversample=False),
            "D: focal loss": dict(loss="focal", oversample=False)}
abl_preds, abl_rows = {}, {}
for name, kw in variants.items():
    abl_preds[name] = train_ensemble(make_model, key, **kw)
    abl_rows[name] = evaluate(y_test, abl_preds[name][1], 0.5)
best_fix = max(["B: oversampling", "C: class weights", "D: focal loss"],
               key=lambda n: average_precision_score(y_val, abl_preds[n][0]))
pv, pt = abl_preds[best_fix]
abl_rows[f"E: {best_fix[3:]} + tuned threshold"] = evaluate(y_test, pt, tune_threshold(y_val, pv))
ablation = pd.DataFrame(abl_rows).T
display(ablation[["PR-AUC", "Recall", "Precision", "F1", "F2", "Threshold", "TP", "FN", "FP"]].round(3))

fig, ax = plt.subplots(figsize=(9, 4))
xs = np.arange(len(ablation)); w = 0.27
for j, (m, col) in enumerate([("Recall", "#c0392b"), ("F1", "#2f6db5"), ("PR-AUC", "#7f8c8d")]):
    ax.bar(xs + (j - 1) * w, ablation[m], w, label=m, color=col)
ax.set_xticks(xs); ax.set_xticklabels([n.split(":")[0] for n in ablation.index])
ax.set_ylim(0, 1.15); ax.set_ylabel("Score on test years (storm class)")
ax.set_title(f"Imbalance ablation · {best_deep}"); ax.legend(ncol=3, loc="upper center", frameon=False)
plt.tight_layout(); plt.savefig(FIG_DIR / "imbalance_ablation.png", dpi=150); plt.show()

## 14. Comparison charts: precision-recall curves and confusion matrix

In [ ]:
model_names = [n for n in preds if not n.startswith(("Always", "Persistence"))]
best_model = max(model_names, key=lambda n: average_precision_score(y_val, preds[n][0]))

fig, ax = plt.subplots(figsize=(7.5, 6.5))
for n in model_names:
    prec, rec, _ = precision_recall_curve(y_test, preds[n][1])
    ax.plot(rec, prec, lw=2.4 if n == best_model else 1.2,
            label=f"{n} (PR-AUC {average_precision_score(y_test, preds[n][1]):.2f})")
ax.axhline(y_test.mean(), color="grey", ls="--", lw=1, label=f"Random guess ({y_test.mean():.2f})")
ax.set_xlabel("Recall (share of storms caught)"); ax.set_ylabel("Precision (share of alerts that were storms)")
ax.set_title("Precision-recall on the test years")
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
plt.tight_layout(); plt.savefig(FIG_DIR / "pr_curves.png", dpi=150); plt.show()

r = results.loc[best_model]
cm = np.array([[r["TN"], r["FP"]], [r["FN"], r["TP"]]]).astype(int)
fig, ax = plt.subplots(figsize=(4.6, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=13,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["No storm", "Storm"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["No storm", "Storm"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"{best_model}\nthreshold {r['Threshold']:.2f}", fontsize=10)
plt.tight_layout(); plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=150); plt.show()

## 15. Business view: alert calendar and cost of errors

Each test day is a hit (storm caught), a miss, a false alarm, or a quiet day correctly left alone. Costs are illustrative: a missed storm is counted as 10 units (e.g. a satellite in the wrong mode, an unprepared grid) and a false alarm as 1 unit (e.g. an unnecessary safe-mode switch). Change `COST_MISSED_STORM` and `COST_FALSE_ALARM` at the top to match a real client.

In [ ]:
thr_best = results.loc[best_model, "Threshold"]
alert = (preds[best_model][1] >= thr_best).astype(int)
dates = data["test"]["date"]
kind = np.select([(alert == 1) & (y_test == 1), (alert == 0) & (y_test == 1), (alert == 1) & (y_test == 0)],
                 ["Hit", "Miss", "False alarm"], "Quiet")
colors = {"Hit": "#27ae60", "Miss": "#c0392b", "False alarm": "#f39c12"}
fig, ax = plt.subplots(figsize=(11, 3))
for k, c in colors.items():
    m = kind == k
    ax.scatter(dates[m], np.full(m.sum(), list(colors).index(k)), s=14, color=c, label=f"{k} ({m.sum()})")
ax.set_yticks(range(3)); ax.set_yticklabels(list(colors)); ax.set_ylim(-0.6, 2.6)
ax.set_title(f"Alert calendar for the test years · {best_model}", pad=24)
ax.legend(ncol=3, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False)
plt.tight_layout(); plt.savefig(FIG_DIR / "alert_calendar.png", dpi=150); plt.show()

def cost(name):
    rr = results.loc[name]
    return rr["FN"] * COST_MISSED_STORM + rr["FP"] * COST_FALSE_ALARM
cost_table = pd.DataFrame({n: {"Missed storms": int(results.loc[n, "FN"]), "False alarms": int(results.loc[n, "FP"]),
                               "Total cost (units)": cost(n)}
                           for n in ["Always 'no storm'", "Persistence (storm today)", "Logistic regression", gb_name, best_model]}).T
cost_table["Saving vs. no warning"] = 1 - cost_table["Total cost (units)"] / cost_table.loc["Always 'no storm'", "Total cost (units)"]
display(cost_table.style.format({"Saving vs. no warning": "{:.0%}", "Total cost (units)": "{:,.0f}"}))

## 16. Save results for the report

In [ ]:
results.to_csv(OUT_DIR / "model_results_test.csv")
ablation.to_csv(OUT_DIR / "imbalance_ablation_test.csv")
cost_table.to_csv(OUT_DIR / "cost_table.csv")
eda.to_csv(OUT_DIR / "events_per_year.csv")

summary = {
    "run_date": datetime.now().isoformat(timespec="minutes"),
    "data_range": [START_DATE, END_DATE],
    "records": {k: len(v) for k, v in raw.items()},
    "storm_onsets": int(len(storms)),
    "class_balance": {s: {"samples": int(len(ix)), "storms": int(y_all[ix].sum()),
                          "storm_share": round(float(y_all[ix].mean()), 4)} for s, ix in sample_idx.items()},
    "event_features": EVENT_FEATURES,
    "images_used": USE_IMAGES,
    "image_coverage": {c: round(float(img_ok[c].mean()), 4) for c in CHANNELS} if USE_IMAGES else None,
    "pretrained_cnn": PRETRAINED_OK,
    "best_model": best_model,
    "ablation_model": best_deep,
    "test_results": results.round(4).to_dict(orient="index"),
    "ablation": ablation.round(4).to_dict(orient="index"),
    "cost_table": cost_table.round(4).to_dict(orient="index"),
}
(OUT_DIR / "results_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("Saved to", OUT_DIR.resolve())
print(json.dumps({k: summary[k] for k in ["best_model", "class_balance", "storm_onsets"]}, indent=2))